# 01 — Data Preparation

Downloads OHLCV data from the Binance API (or another source), converts it to the standard CSV format, and uploads it to the `epochquant-training` GCS bucket.


In [ ]:
# ── Install dependencies (first run only) ──────────────────────
# !pip install -r ../requirements.txt


In [ ]:
import sys
sys.path.insert(0, '..')  # repo root

GCS_BUCKET  = 'epochquant-training'
GCS_PROJECT = None  # Set to your GCP project ID if needed
SYMBOL      = 'BNBUSDT'
TIMEFRAME   = '1m'

GCS_RAW_PATH       = f'gs://{GCS_BUCKET}/raw/{SYMBOL.lower()}/'
GCS_PROCESSED_PATH = f'gs://{GCS_BUCKET}/processed/{SYMBOL.lower()}_{TIMEFRAME}.csv'

print(f'Symbol   : {SYMBOL}')
print(f'Timeframe: {TIMEFRAME}')
print(f'GCS out  : {GCS_PROCESSED_PATH}')


In [ ]:
# ── Option A: Load from existing local JSON files ───────────────
import sys; sys.path.insert(0, '..')
from data.data_loader import load_dataset

LOCAL_JSON_DIR = f'../data/raw/{SYMBOL.lower()}/'
df = load_dataset(LOCAL_JSON_DIR)
print(f'Loaded {len(df):,} rows  ({df["timestamps"].min()} → {df["timestamps"].max()})')
df.head()


In [ ]:
# ── Basic data quality checks ───────────────────────────────────
print('Shape     :', df.shape)
print('NaN count :', df.isna().sum().sum())
print('Date range:', df['timestamps'].min(), '→', df['timestamps'].max())
print('\nStats:')
df[['open','high','low','close','volume']].describe()


In [ ]:
# ── Save processed CSV to GCS ───────────────────────────────────
import gcsfs

storage_opts = {'project': GCS_PROJECT} if GCS_PROJECT else {}
df.to_csv(GCS_PROCESSED_PATH, index=False, storage_options=storage_opts)
print(f'Saved to: {GCS_PROCESSED_PATH}')


In [ ]:
# ── Verify upload by reading back ───────────────────────────────
from data.data_loader import load_from_gcs
df_check = load_from_gcs(GCS_PROCESSED_PATH, gcs_project=GCS_PROJECT)
print(f'Verified: {len(df_check):,} rows read back from GCS')
